In [1]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value
os.environ['HF_HOME'] = "/root/autodl-tmp/.cache/huggingface"

# PEFT 库 QLoRA 实战 - ChatGLM3-6B

通常，模型被量化后不会进一步训练用于下游任务，因为由于权重和激活的较低精度，训练可能不稳定。

但是由于PEFT方法只添加额外的可训练参数，这使得我们可以使用PEFT适配器（Adapter）来训练一个量化模型！将量化与PEFT结合起来可以成为在单个GPU上训练大模型的微调策略。

例如，`QLoRA` 是一种将模型量化为4位然后使用LoRA进行训练的方法，使得在单个16GB GPU（本教程以 NVIDIA T4为例）上微调一个具有65B参数的大模型成为可能。

THUDM Hugging Face 主页：https://huggingface.co/THUDM

## 教程说明

本教程使用 QLoRA 论文中介绍的量化技术：`NF4 数据类型`、`双量化` 和 `混合精度计算`，在 `ChatGLM3-6b` 模型上实现了 QLoRA 微调。并展示了完整的 QLoRA 微调流程，具体如下：

- 数据准备
    - 下载数据集
    - 设计 Tokenizer 函数处理样本（map、shuffle、flatten）
    - 自定义批量数据处理类 DataCollatorForChatGLM
- 训练模型
    - 加载 ChatGLM3-6B 量化模型
    - PEFT 量化模型预处理（prepare_model_for_kbit_training）
    - QLoRA 适配器配置（TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING）
    - 微调训练超参数配置（TrainingArguments）
    - 开启训练（trainer.train)
    - 保存QLoRA模型（trainer.model.save_pretrained)
- [模型推理](peft_chatglm_inference.ipynb)
    - 加载 ChatGLM3-6B 基础模型
    - 加载 ChatGLM3-6B QLoRA 模型（PEFT Adapter）
    - 微调前后对比

In [2]:
# 定义全局变量和参数
model_name_or_path = 'THUDM/chatglm3-6b'  # 模型ID或本地路径
train_data_path = 'HasturOfficial/adgen'    # 训练数据路径
eval_data_path = None                     # 验证数据路径，如果没有则设置为None
seed = 8                                 # 随机种子
max_input_length = 512                    # 输入的最大长度
max_output_length = 1536                  # 输出的最大长度
lora_rank = 4                             # LoRA秩
lora_alpha = 32                           # LoRA alpha值
lora_dropout = 0.05                       # LoRA Dropout率
resume_from_checkpoint = None             # 如果从checkpoint恢复训练，指定路径
prompt_text = ''                          # 所有数据前的指令文本
compute_dtype = 'fp32'                    # 计算数据类型（fp32, fp16, bf16）

## 数据准备

### 下载数据集

从 Hugging Face 加载 adgen 数据集，并tokenize，shuffle

In [3]:
from datasets import load_dataset

dataset = load_dataset(train_data_path)

In [4]:
from typing import Union
from datasets import Dataset, DatasetDict
import pandas as pd

def inspect_dataset(
    ds: Union[Dataset, DatasetDict],
    num_samples: int = 5,
    seed: int = 42
) -> None:
    """
    快速查看 Hugging Face Dataset 或 DatasetDict 的结构与示例。

    参数
    ----
    ds : Dataset 或 DatasetDict
        要检查的数据集（可以是单个 split 的 Dataset，也可以是多个 split 的 DatasetDict）。
    num_samples : int, optional (default=5)
        每个 split 中要随机抽看的示例条目数。
    seed : int, optional (default=42)
        随机种子，用于可复现地抽样示例。

    输出
    ----
    该函数会依次打印每个 split 的：
      1. split 名称（对于 Dataset 则视为 'default'）
      2. 总样本数
      3. 所有字段（columns）及其类型
      4. 随机抽取的若干行示例（以 pandas.DataFrame 形式展示）
    """
    def _inspect_split(name: str, dset: Dataset):
        print(f"\n--- Split: {name} ---")
        print(f"样本数：{len(dset)}")
        print("字段与类型:")
        for col, feat in dset.features.items():
            print(f"  - {col}: {feat}")
        # 随机抽样并展示
        sample_idx = dset.shuffle(seed=seed).select(range(min(num_samples, len(dset))))
        df = pd.DataFrame(sample_idx)
        display(df)   # 在 Notebook 中可直接展示；普通脚本可改成 print(df.head())

    # 如果是 DatasetDict，遍历每个 split
    if isinstance(ds, DatasetDict):
        for split_name, split_ds in ds.items():
            _inspect_split(split_name, split_ds)
    # 如果是单一的 Dataset
    else:
        _inspect_split("default", ds)

inspect_dataset(dataset)


--- Split: train ---
样本数：114599
字段与类型:
  - content: Value(dtype='string', id=None)
  - summary: Value(dtype='string', id=None)


,content,summary
0,类型#裙*版型#宽松*风格#复古*风格#简约*图案#抽象*图案#复古*图案#线条*图案#刺绣...,圆领设计的上衣，修饰颈部的线条，时尚百搭。抽象人脸图案，艺术玩味俏皮，凸显个性的同时，体现独...
1,类型#裤*风格#运动*图案#字母*图案#文字*裤款式#抽绳,舒适面料，穿着亲肤有型。抽绳设计的休闲裤版型，运动自如，展现型男青春活力的风范。裤身上，印有...
2,类型#裙*版型#显瘦*图案#线条*图案#撞色*裙型#a字*裙衣门襟#拉链*裙款式#木耳*裙款...,这款背心裙采用了局部撞色的设计，从视觉上起到了让人眼前一亮的效果。木耳领口的设计别致时髦，修...
3,类型#上衣*材质#牛仔布*材质#水洗*颜色#蓝色*风格#青春*风格#清新*图案#刺绣*衣样式...,吸引我的是外套上的装饰，不会轻易撞衫！工整对称口袋，重工钉珠、褶皱花边勾勒，很有新鲜感的少女...
4,类型#上衣*风格#性感*衣样式#毛衣*衣领型#v领,深v的设计把性感与诱惑完美的诠释出来，作为女性，很适合拥有这样一件既能保暖，又能提升气质的毛...



--- Split: validation ---
样本数：1070
字段与类型:
  - content: Value(dtype='string', id=None)
  - summary: Value(dtype='string', id=None)


,content,summary
0,类型#裙*材质#雪纺*裙长#连衣裙*裙领型#v领*裙袖型#喇叭袖*裙衣门襟#系带,女人和雪纺仿佛天生就有一种不解之缘。一见钟情，<UNK>倾心。这款雪纺连衣裙设计了优雅的系带...
1,类型#裙*材质#牛仔布*风格#简约*风格#青春*图案#卡通*图案#刺绣*裙型#牛仔裙*裙衣门...,牛仔裤是衣橱里一年四季都不可或缺的时尚单品，这款就是比较简约版型的，整体优选的牛仔棉弹面料更...
2,类型#裙*材质#网纱*材质#蕾丝*图案#蕾丝*裙长#长裙*裙长#半身裙*裙款式#拼接*裙款式...,网纱拼接半身裙精选优质网纱质地轻盈、触感柔韧、垂感自然飘逸。拼接精致的镂空蕾丝，轻薄柔软。精...
3,类型#上衣*版型#显瘦*颜色#白色*颜色#纯色*图案#纯色*图案#碎花*衣样式#衬衫*衣袖长...,推荐这款来自时尚服装品牌tedbaker的长袖衬衫。这款衬衫采用纯色色调搭载小碎花的点缀设计...
4,类型#裙*版型#显瘦*图案#线条*裙款式#勾花镂空*裙款式#收腰,以修身收腰的版型勾勒出优雅的裙装，展现出纤细的腰身和修长的腿部线条。在清爽镂空的面料上加入别...


# ChatGLM3 中的标签处理与损失计算解析

在训练像 ChatGLM3-6B 这样的语言模型时，输入(input_ids)和标签(labels)的处理方式对于正确计算损失至关重要。下面详细解释这个过程：

## 1. 输入序列(input_ids)的填充处理

- **目的**：确保批处理中所有序列长度一致
- **方法**：使用特定的填充标记(通常是PAD_TOKEN_ID)将较短的序列填充到指定长度
- **特点**：这些填充标记被添加到序列末尾，但不改变原始文本的语义内容

## 2. 标签序列(labels)的填充处理

在对话模型中，标签序列通常包含两部分：
- 输入部分(用户查询)
- 输出部分(模型应生成的回答)

标签处理有两个关键点：

### A. 输入部分的处理
- 将输入部分的所有标记ID设置为 `-100`
- 当值为 `-100` 时，PyTorch的损失函数会**完全忽略**这些位置
- 这确保模型**不会**因为重复输入内容而受到惩罚或奖励

### B. 输出部分的处理
- 保留原始标记ID
- 这些位置会参与损失计算
- 模型生成的内容与这些标记进行比较，计算交叉熵损失

## 3. 为什么这样处理？

- **只评估生成能力**：我们只关心模型是否能正确生成回答，而不是复述输入
- **避免混淆学习信号**：如果不忽略输入部分，模型会收到"应该复制输入"的错误信号
- **专注于真正的任务**：语言模型的核心任务是基于上下文生成合适的下一个标记

## 4. 示例说明

假设有对话：
- 输入："你好，请介绍自己"
- 输出："我是ChatGLM3，一个AI助手"

标签处理后：
- labels = [-100, -100, -100, -100, ..., ID("我"), ID("是"), ID("ChatGLM3"), ...]

损失计算只考虑输出部分的标记，忽略所有值为 `-100` 的位置。

# 交叉熵损失计算详解：标签修改(-100)的作用与影响

## 1. 交叉熵损失基础

在语言模型训练中，交叉熵损失函数用于衡量模型预测与真实标签之间的差异：

$$\text{Loss} = -\frac{1}{n}\sum_{i=1}^{n}\sum_{c=1}^{C}y_{i,c}\log(p_{i,c})$$

其中：
- $n$ 是序列长度
- $C$ 是词表大小
- $y_{i,c}$ 是位置 $i$ 处的真实标签的one-hot编码
- $p_{i,c}$ 是模型在位置 $i$ 预测类别 $c$ 的概率

对于语言模型，这简化为：

$$\text{Loss} = -\frac{1}{n}\sum_{i=1}^{n}\log(p_{i,y_i})$$

其中 $p_{i,y_i}$ 是模型在位置 $i$ 预测真实标签 $y_i$ 的概率。

## 2. 不修改标签时的损失计算

假设我们有这样一个对话样本：
- 输入(query)："你好"
- 输出(answer)："我是ChatGLM3"

标记化后（简化表示）：
- query_ids = [101, 1, 2, 102]  # [CLS, "你", "好", SEP]
- answer_ids = [3, 4, 5, 6, 7]  # ["我", "是", "Chat", "GLM", "3"]

完整序列：[101, 1, 2, 102, 3, 4, 5, 6, 7]

**不修改标签时**:
```python
labels = [101, 1, 2, 102, 3, 4, 5, 6, 7]  # 整个序列都参与损失计算

# 损失计算（伪代码）
total_loss = 0
for i in range(len(labels)):
    # 计算模型在位置i预测标签labels[i]的负对数概率
    loss_at_position = -log(model_outputs[i][labels[i]])
    total_loss += loss_at_position

average_loss = total_loss / len(labels)  # 计算平均损失
````

**问题**：

1. 模型被强制学习预测输入部分(前4个标记)，这是没有意义的
2. 输入部分的损失会影响整体优化方向
3. 导致模型学习"复制输入"而非"理解输入并生成回答"

## 3. 修改标签为-100后的损失计算

**修改标签时**:

```python
labels = [-100, -100, -100, -100, 3, 4, 5, 6, 7]  # 只有输出部分参与损失计算

# 损失计算（伪代码）
total_loss = 0
valid_positions = 0
for i in range(len(labels)):
    if labels[i] == -100:
        continue  # 跳过标记为-100的位置
    
    # 计算模型在位置i预测标签labels[i]的负对数概率
    loss_at_position = -log(model_outputs[i][labels[i]])
    total_loss += loss_at_position
    valid_positions += 1

average_loss = total_loss / valid_positions  # 只计算有效位置的平均损失
```

## 4. 具体对比示例

假设模型在各位置的预测概率如下（简化数字）：

| 位置 | 预测正确标记的概率   |
| -- | ----------- |
| 0  | 0.9 (预测101) |
| 1  | 0.8 (预测1)   |
| 2  | 0.7 (预测2)   |
| 3  | 0.9 (预测102) |
| 4  | 0.6 (预测3)   |
| 5  | 0.5 (预测4)   |
| 6  | 0.4 (预测5)   |
| 7  | 0.3 (预测6)   |
| 8  | 0.2 (预测7)   |

**不修改标签时的损失**:

```
Loss = -(log(0.9) + log(0.8) + log(0.7) + log(0.9) + log(0.6) + log(0.5) + log(0.4) + log(0.3) + log(0.2)) / 9
Loss = -((-0.105) + (-0.223) + (-0.357) + (-0.105) + (-0.511) + (-0.693) + (-0.916) + (-1.204) + (-1.609)) / 9
Loss = 0.636
```

**修改标签时的损失**:

```
Loss = -(log(0.6) + log(0.5) + log(0.4) + log(0.3) + log(0.2)) / 5
Loss = -((-0.511) + (-0.693) + (-0.916) + (-1.204) + (-1.609)) / 5
Loss = 0.987
```

## 5. 关键区别与优势

| 方面     | 不修改标签               | 修改标签为-100    |
| ------ | ------------------- | ------------ |
| 损失计算范围 | 整个序列                | 仅输出部分        |
| 优化目标   | 混合了"复制输入"和"生成输出"    | 专注于"生成合适的输出" |
| 损失值    | 通常较小(被输入部分的高概率预测拉低) | 更准确反映生成能力    |
| 学习效果   | 可能导致模型过度关注输入复制      | 专注于提高生成质量    |

通过将输入部分的标签设为-100，我们确保模型：

1. 不会浪费参数学习"复制输入"这种简单任务
2. 将全部注意力放在提高生成能力上
3. 损失值更准确地反映模型的真正生成能力

这种标签处理方式是训练高质量对话模型的关键技术之一。



In [5]:
from transformers import AutoTokenizer

# revision='b098244' 版本对应的 ChatGLM3-6B 设置 use_reentrant=False
# 最新版本 use_reentrant 被设置为 True，会增加不必要的显存开销
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path,
                                          trust_remote_code=True,
                                          revision='b098244')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/root/miniconda3/lib/python3.12/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
# tokenize_func 函数
def tokenize_func(example, tokenizer, ignore_label_id=-100):
    """
    对单个数据样本进行tokenize处理。

    参数:
    example (dict): 包含'content'和'summary'键的字典，代表训练数据的一个样本。
    tokenizer (transformers.PreTrainedTokenizer): 用于tokenize文本的tokenizer。
    ignore_label_id (int, optional): 在label中用于填充的忽略ID，默认为-100。

    返回:
    dict: 包含'tokenized_input_ids'和'labels'的字典，用于模型训练。
    """

    # 构建问题文本
    question = prompt_text + example['content']
    if example.get('input', None) and example['input'].strip():
        question += f'\n{example["input"]}'

    # 构建答案文本
    answer = example['summary']

    # 对问题和答案文本进行tokenize处理
    q_ids = tokenizer.encode(text=question, add_special_tokens=False)
    a_ids = tokenizer.encode(text=answer, add_special_tokens=False)

    # 如果tokenize后的长度超过最大长度限制，则进行截断
    if len(q_ids) > max_input_length - 2:  # 保留空间给gmask和bos标记
        q_ids = q_ids[:max_input_length - 2]
    if len(a_ids) > max_output_length - 1:  # 保留空间给eos标记
        a_ids = a_ids[:max_output_length - 1]

    # 构建模型的输入格式
    input_ids = tokenizer.build_inputs_with_special_tokens(q_ids, a_ids)
    question_length = len(q_ids) + 2  # 加上gmask和bos标记

    # 构建标签，对于问题部分的输入使用ignore_label_id进行填充
    labels = [ignore_label_id] * question_length + input_ids[question_length:]

    return {'input_ids': input_ids, 'labels': labels}

这里的核心在于「教师强制（teacher forcing）」和因果语言模型的训练方式：

1. **为什么输入要包含 question 和 answer？**

   * 对于自回归（causal）语言模型（如 GPT 这类），在训练时我们需要把上下文（也就是你的 question）和目标输出（answer）拼成一个长序列，一次性丢给模型。
   * 模型在每个位置都会计算下一个 token 的预测概率。把 question 和 answer 拼在一起，模型就能学习「在完整上下文＋之前已经生成的 answer 部分」下，预测下一个 answer token。

2. **为什么 labels 也要包含 question 和 answer？**

   * 其实在计算 loss 时，question 部分的 labels 都被设为 `-100`（或其他 `ignore_index`），这样 CrossEntropyLoss 会跳过它们，不计算梯度。
   * 真正参与损失的，只是后半段 answer 部分。这种做法让模型专注于「给定 question，上下文＋已生成 answer，生成下一个 answer token」的能力，而不是去学「复制 question」。

3. **训练流程到底是怎样的？**

   * **拼序列**：`[ question_tokens..., answer_tokens... ]`
   * **输入 input\_ids**：直接就是上面这个完整序列。
   * **构造 labels**：前面 question 部分打上 `-100`，后面 answer 部分打上它们自己的 token id。
   * **前向计算**：模型在每个位置都会给出一个对下一个 token 的分布。
   * **计算 loss**：CrossEntropyLoss 只对那些 label ≠ `-100` 的位置（也就是 answer 部分）计算负对数似然。
   * **反向传播**：更新模型参数，让它更擅长在相同 context 下生成正确的 answer。

4. **你为什么要「把问题和答案都告诉」模型？**

   * **给定上下文才能预测下文**：语言模型本质是「在已有文本后面接着写」。如果只给 answer 而不提供 question，模型就没法学到「看到 question 时应该往哪里接」，也无法做条件生成。
   * **teacher forcing**：训练时我们用「真答案」来强制喂给模型，而不是让它自回归地、一个一个地预测并把预测结果再当输入。这样收敛更快、更稳定。

---

**要点总结**

* 输入 `input_ids` 包含 question + answer，是为了让模型在完整上下文下学习预测 answer。
* labels 虽然也包含 question + answer，但我们把 question 部分标成 `-100`，从而只对 answer 部分计算损失。
* 这就是典型的自回归条件生成（conditional generation）训练流程，你无需「单独」告诉模型 question、再「单独」给它 answer；只要把它们拼在一起，并在计算 loss 时跳过 question 部分，就能同时满足「提供上下文」和「计算 answer 的预测误差」两者需求。


In [8]:
# pip install transformers==4.40.2 不然会在padding报错
column_names = dataset['train'].column_names
tokenized_dataset = dataset['train'].map(
    lambda example: tokenize_func(example, tokenizer),
    batched=False, 
    remove_columns=column_names
)

Map:   0%|          | 0/114599 [00:00<?, ? examples/s]

In [9]:
inspect_dataset(tokenized_dataset,num_samples=2)


--- Split: default ---
样本数：114599
字段与类型:
  - input_ids: Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None)
  - labels: Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None)


,input_ids,labels
0,"[64790, 64792, 30910, 33467, 31010, 56778, 309...","[-100, -100, -100, -100, -100, -100, -100, -10..."
1,"[64790, 64792, 30910, 33467, 31010, 56532, 309...","[-100, -100, -100, -100, -100, -100, -100, -10..."


### 数据集处理：shuffle & flatten 

洗牌(shuffle)会将数据集的索引列表打乱，以创建一个索引映射。

然而，一旦您的数据集具有索引映射，速度可能会变慢10倍。这是因为需要额外的步骤来使用索引映射获取要读取的行索引，并且最重要的是，您不再连续地读取数据块。

要恢复速度，需要再次使用 Dataset.flatten_indices()将整个数据集重新写入磁盘上，从而删除索引映射。

ref: https://huggingface.co/docs/datasets/v2.15.0/en/package_reference/main_classes#datasets.Dataset.flatten_indices

In [10]:
tokenized_dataset = tokenized_dataset.shuffle(seed=seed)

In [11]:
tokenized_dataset = tokenized_dataset.flatten_indices()

Flattening the indices:   0%|          | 0/114599 [00:00<?, ? examples/s]

### 定义 DataCollatorForChatGLM 类 批量处理数据

这个 `DataCollatorForChatGLM` 类的作用是——在训练或推理时，将多条已经 **tokenize** 过的样本（每条样本包含 `input_ids` 和对应的 `labels` 列表）整理成一个等长的批量张量（`torch.Tensor`），并在必要时完成 **padding** 或 **truncation**。下面按结构分步解释：

1. **初始化 (`__init__`)**

   * `pad_token_id`：用于填充 `input_ids` 的 token ID。
   * `max_length`：批量中任何一条样本在做完 padding 之后的最大允许长度，默认 2048。
   * `ignore_label_id`：用于填充 `labels` 时的填充值，通常设为 -100，让损失计算时自动忽略这些位置。

2. **调用接口 (`__call__`)**

   * 输入：`batch_data`，一个列表，列表中每个元素都是字典，包含两个键：

     * `'input_ids'`：模型输入的 token ID 序列
     * `'labels'`：训练目标序列，其中问题部分被填成 `ignore_label_id`，答案部分是真实标签
   * 处理步骤：

     1. **计算各样本长度**：遍历 `batch_data`，记录每条 `input_ids` 的长度，取最大值 `batch_max_len`。
     2. **按长度降序排序**（`sorted(..., key=lambda x: -x[0])`）：先处理长的样本，有利于某些后端加速（比如 RNN pack）。
     3. **Padding / Truncation**

        * 对每条样本：

          * 计算 `pad_len = batch_max_len - actual_len`
          * 在 `input_ids` 后面追加 `pad_token_id` 重复 `pad_len` 次；
          * 在 `labels` 后面追加 `ignore_label_id` 重复 `pad_len` 次。
        * 如果 `batch_max_len` 本身超过了 `self.max_length`，就再把它们截断到 `self.max_length`。
     4. **构造张量**

        * 把所有处理完的 `ids` 列表和 `label` 列表分别转成 `torch.LongTensor`，再用 `torch.stack` 堆叠成形状 `(batch_size, seq_len)` 的二维张量。
   * 输出：一个字典，包含两个键：

     * `'input_ids'`：形状 `[batch_size, seq_len]` 的长整型张量，用作模型输入。
     * `'labels'`：同样形状的张量，用于计算损失，padding 部分因值为 `ignore_label_id` 而被忽略。

3. **设计要点**

   * **动态最大长度**：按每个 batch 内最长样本动态 padding，避免无谓的过度填充。
   * **可控的全局上限**：`self.max_length` 保证任何极端情况下都不会超出内存或显存承受范围。
   * **排序优化**：降序排序便于一些模型或框架（如 RNN/LSTM）使用打包序列（`pack_padded_sequence`）时更高效。
   * **标签填充忽略**：`ignore_label_id=-100` 与 PyTorch 的交叉熵损失函数默认配置匹配，可自动跳过填充部分。

这样，在训练或评估时，只要把 `DataCollatorForChatGLM(pad_token_id, max_length)` 传给 DataLoader，就可以自动地把不定长的样本列表转换为一个规则的、可直接输入模型的张量批次。


In [12]:
import torch
from typing import List, Dict, Optional

# DataCollatorForChatGLM 类
class DataCollatorForChatGLM:
    """
    用于处理批量数据的DataCollator，尤其是在使用 ChatGLM 模型时。

    该类负责将多个数据样本（tokenized input）合并为一个批量，并在必要时进行填充(padding)。

    属性:
    pad_token_id (int): 用于填充(padding)的token ID。
    max_length (int): 单个批量数据的最大长度限制。
    ignore_label_id (int): 在标签中用于填充的ID。
    """

    def __init__(self, pad_token_id: int, max_length: int = 2048, ignore_label_id: int = -100):
        """
        初始化DataCollator。

        参数:
        pad_token_id (int): 用于填充(padding)的token ID。
        max_length (int): 单个批量数据的最大长度限制。
        ignore_label_id (int): 在标签中用于填充的ID，默认为-100。
        """
        self.pad_token_id = pad_token_id
        self.ignore_label_id = ignore_label_id
        self.max_length = max_length

    def __call__(self, batch_data: List[Dict[str, List]]) -> Dict[str, torch.Tensor]:
        """
        处理批量数据。

        参数:
        batch_data (List[Dict[str, List]]): 包含多个样本的字典列表。

        返回:
        Dict[str, torch.Tensor]: 包含处理后的批量数据的字典。
        """
        # 计算批量中每个样本的长度
        len_list = [len(d['input_ids']) for d in batch_data]
        batch_max_len = max(len_list)  # 找到最长的样本长度

        input_ids, labels = [], []
        for len_of_d, d in sorted(zip(len_list, batch_data), key=lambda x: -x[0]):
            pad_len = batch_max_len - len_of_d  # 计算需要填充的长度
            # 添加填充，并确保数据长度不超过最大长度限制
            ids = d['input_ids'] + [self.pad_token_id] * pad_len
            label = d['labels'] + [self.ignore_label_id] * pad_len
            if batch_max_len > self.max_length:
                ids = ids[:self.max_length]
                label = label[:self.max_length]
            input_ids.append(torch.LongTensor(ids))
            labels.append(torch.LongTensor(label))

        # 将处理后的数据堆叠成一个tensor
        input_ids = torch.stack(input_ids)
        labels = torch.stack(labels)

        return {'input_ids': input_ids, 'labels': labels}


In [13]:
# 准备数据整理器
data_collator = DataCollatorForChatGLM(pad_token_id=tokenizer.pad_token_id)

## 训练模型

### 加载 ChatGLM3-6B 量化模型

使用 `nf4` 量化数据类型加载模型，开启双量化配置，以`bf16`混合精度训练，预估显存占用接近4GB

In [14]:
from transformers import AutoModel, BitsAndBytesConfig

_compute_dtype_map = {
    'fp32': torch.float32,
    'fp16': torch.float16,
    'bf16': torch.bfloat16
}

# QLoRA 量化配置
q_config = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=_compute_dtype_map['bf16'])


### 加载模型


In [ ]:
# revision='b098244' 版本对应的 ChatGLM3-6B 设置 use_reentrant=False
# 最新版本 use_reentrant 被设置为 True，会增加不必要的显存开销
model = AutoModel.from_pretrained(model_name_or_path,
                                  quantization_config=q_config,
                                  device_map='auto',
                                  trust_remote_code=True,
                                  revision='b098244')

/root/miniconda3/lib/python3.12/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

configuration_chatglm.py:   0%|          | 0.00/2.33k [00:00<?, ?B/s]

modeling_chatglm.py:   0%|          | 0.00/55.7k [00:00<?, ?B/s]

quantization.py:   0%|          | 0.00/14.7k [00:00<?, ?B/s]

...

model-00001-of-00007.safetensors:   0%|          | 0.00/1.83G [00:00<?, ?B/s]

model-00002-of-00007.safetensors:   0%|          | 0.00/1.97G [00:00<?, ?B/s]

model-00003-of-00007.safetensors:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

model-00004-of-00007.safetensors:   0%|          | 0.00/1.82G [00:00<?, ?B/s]

model-00005-of-00007.safetensors:   0%|          | 0.00/1.97G [00:00<?, ?B/s]

In [ ]:
# 获取当前模型占用的 GPU显存（差值为预留给 PyTorch 的显存）
memory_footprint_bytes = model.get_memory_footprint()
memory_footprint_mib = memory_footprint_bytes / (1024 ** 2)  # 转换为 MiB

print(f"{memory_footprint_mib:.2f}MiB")

### 预处理量化模型

预处理量化后的模型，使其可以支持低精度微调训练

ref: https://huggingface.co/docs/peft/main/en/developer_guides/quantization#quantize-a-model

In [ ]:
from peft import TaskType, LoraConfig, get_peft_model, prepare_model_for_kbit_training

kbit_model = prepare_model_for_kbit_training(model)

### 自定义模型新增 Adapter 

当新的热门 transformer 网络架构（新模型）发布时，Huggingface 社区会尽力快速将它们添加到PEFT中。

如果是 Hugging Face Transformers 库还未内置支持的模型，可以使用自定义模型的方式进行配置。

具体来说，在初始化相应的微调配置类（例如`LoraConfig`）时，我们需要显式指定在哪些层新增适配器（Adapter），并将其设置正确。

ref: https://huggingface.co/docs/peft/developer_guides/custom_models


#### PEFT 适配模块设置


在PEFT库的 [constants.py](https://github.com/huggingface/peft/blob/main/src/peft/utils/constants.py) 文件中定义了不同的 PEFT 方法，在各类大模型上的微调适配模块。

通常，名称相同的模型架构也类似，应用微调方法时的适配器设置也几乎一致。

例如，如果新模型架构是`mistral`模型的变体，并且您想应用 LoRA 微调。在 TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING中`mistral`包含["q_proj", "v_proj"]。

这表示说，对于`mistral`模型，LoRA 的 target_modules 通常是 ["q_proj", "v_proj"]。

In [ ]:
from peft.utils import TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING

target_modules = TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING['chatglm']

In [ ]:
target_modules

### LoRA 适配器配置

In [ ]:
lora_config = LoraConfig(
    target_modules=target_modules,
    r=lora_rank,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias='none',
    inference_mode=False,
    task_type=TaskType.CAUSAL_LM
)

In [ ]:
qlora_model = get_peft_model(kbit_model, lora_config)

In [ ]:
qlora_model.print_trainable_parameters()

### 训练超参数配置

- 1个epoch表示对训练集的所有样本进行一次完整的训练。
- `num_train_epochs` 表示要完整进行多少个 epochs 的训练。

#### 关于使用 num_train_epochs 时，训练总步数 `steps` 的计算方法

- 训练总步数： `total_steps = steps/epoch * num_train_epochs` 
- 每个epoch的训练步数：`steps/epoch = num_train_examples / (batch_size * gradient_accumulation_steps)`


**以 `adgen` 数据集为例计算**

```json
DatasetDict({
    train: Dataset({
        features: ['content', 'summary'],
        num_rows: 114599
    })
    validation: Dataset({
        features: ['content', 'summary'],
        num_rows: 1070
    })
})
```

代入超参数和配置进行计算：

```python
num_train_epochs = 1
num_train_examples = 114599
batch_size = 16
gradient_accumulation_steps = 4


steps = num_train_epochs * num_train_examples / (batch_size * gradient_accumulation_steps)
      = 1 * 114599 / (16 * 4)
      = 1790
```

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=f"models/{model_name_or_path}",          # 输出目录
    per_device_train_batch_size=16,                     # 每个设备的训练批量大小
    gradient_accumulation_steps=4,                     # 梯度累积步数
    # per_device_eval_batch_size=8,                      # 每个设备的评估批量大小
    learning_rate=1e-3,                                # 学习率
    num_train_epochs=1,                                # 训练轮数
    lr_scheduler_type="linear",                        # 学习率调度器类型
    warmup_ratio=0.1,                                  # 预热比例
    logging_steps=10,                                 # 日志记录步数
    save_strategy="steps",                             # 模型保存策略
    save_steps=200,                                    # 模型保存步数
    # evaluation_strategy="steps",                       # 评估策略
    # eval_steps=500,                                    # 评估步数
    optim="adamw_torch",                               # 优化器类型
    fp16=True,                                        # 是否使用混合精度训练
)


In [ ]:
trainer = Trainer(
        model=qlora_model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator
    )

#### 训练参数（用于演示）

In [24]:
from transformers import TrainingArguments, Trainer

training_demo_args = TrainingArguments(
    output_dir=f"models/demo/{model_name_or_path}",          # 输出目录
    per_device_train_batch_size=16,                     # 每个设备的训练批量大小
    gradient_accumulation_steps=4,                     # 梯度累积步数
    learning_rate=1e-3,                                # 学习率
    max_steps=100,                                     # 训练步数
    lr_scheduler_type="linear",                        # 学习率调度器类型
    warmup_ratio=0.1,                                  # 预热比例
    logging_steps=10,                                 # 日志记录步数
    save_strategy="steps",                             # 模型保存策略
    save_steps=20,                                    # 模型保存步数
    optim="adamw_torch",                               # 优化器类型
    fp16=True,                                        # 是否使用混合精度训练
)

In [26]:
trainer = Trainer(
        model=qlora_model,
        args=training_demo_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator
    )

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


### 开始训练


In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained(f"models/demo/{model_name_or_path}")